In [ ]:
# ============================================================
# PII Detection with Piiranha-v1
# Google Colab Template
# ============================================================

# ── 1. Install dependencies ──────────────────────────────────
!pip install transformers torch -q

# ── 2. Imports ───────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch

# ── 3. Load model ────────────────────────────────────────────
model_name = "iiiorg/piiranha-v1-detect-personal-information"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Move to GPU if available
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

# ── 4. Create pipeline ───────────────────────────────────────
pii_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",  # merges subword tokens into full words
    device=device
)

# ── 5. Helper: redact PII from text ──────────────────────────
def redact_pii(text, replacement="[REDACTED]"):
    results = pii_pipeline(text)
    
    # Sort by start position (reverse) so we replace from end to start
    # This way character positions don't shift as we replace
    results = sorted(results, key=lambda x: x["start"], reverse=True)
    
    for entity in results:
        start = entity["start"]
        end   = entity["end"]
        label = entity["entity_group"]
        text  = text[:start] + f"[{label}]" + text[end:]
    
    return text

def show_entities(text):
    results = pii_pipeline(text)
    print(f"Input:  {text}")
    print(f"Found {len(results)} PII entities:\n")
    for e in results:
        print(f"  {e['entity_group']:<20} → '{e['word']}' (confidence: {e['score']:.2%})")

# ── 6. Chunk helper (context limit is 256 tokens) ────────────
def redact_long_text(text, replacement="[REDACTED]", chunk_size=200):
    """Split text into sentences/chunks and redact each one."""
    import re
    # Split on sentence boundaries
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    chunks, current = [], ""
    for sentence in sentences:
        if len(current) + len(sentence) < chunk_size * 4:  # rough char estimate
            current += " " + sentence
        else:
            if current:
                chunks.append(current.strip())
            current = sentence
    if current:
        chunks.append(current.strip())
    
    return " ".join(redact_pii(chunk) for chunk in chunks)


# ── 7. Try it out ─────────────────────────────────────────────
sample = """
Hello, my name is Pascal Müller and I live at Bahnhofstrasse 10, Zurich.
You can reach me at pascal.mueller@example.com or call +41 79 123 45 67.
My date of birth is 15.03.1971 and my IBAN is CH56 0483 5012 3456 7800 9.
"""

print("=" * 60)
print("ENTITY DETECTION")
print("=" * 60)
show_entities(sample)

print("\n" + "=" * 60)
print("REDACTED OUTPUT")
print("=" * 60)
print(redact_pii(sample))